In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen imp

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\


In [2]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [4]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

In [5]:
from training.cross_validation import create_test_fold_indices
labels_df = create_test_fold_indices(labels_df, 5)

In [8]:
labels_df["Site_Day"] = labels_df["Site"] + "_" + pd.to_datetime(labels_df["clip_start_time"], format="mixed").dt.strftime("%Y-%m-%d")

In [9]:
labels_df[labels_df["Boat"]==1]["Site_Day"].unique()

<StringArray>
['BSM_2017-07-24', 'BSM_2017-07-25', 'BSM_2017-07-29', 'BSM_2017-08-01',
 'BSM_2017-08-08', 'BSM_2017-08-12', 'CAC_2021-07-14', 'CAC_2021-07-18',
 'CAC_2021-08-04', 'KAM_2020-07-26', 'KAM_2020-07-29', 'KAM_2020-07-30',
 'KAM_2020-07-31', 'KAM_2020-08-01', 'KAM_2021-07-20', 'KAM_2021-07-23',
 'KAM_2021-07-27', 'KAM_2021-07-28', 'KAM_2021-08-01', 'KAM_2021-08-05',
 'KAM_2021-08-09', 'RDL_2020-07-22', 'RDL_2020-07-30', 'RDL_2020-08-28']
Length: 24, dtype: str

In [15]:
pd.set_option('display.max_columns', None)
labels_df.head(2)

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s,ClipFilenamePt,test_fold_idx,Site_Day
0,BSM_20170724_09471700.wav,0,0,0,0,0,1.0,evaluation_v2,NaN,0.0,1.0,a,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017\Boat,201359382,-172.7,2017-07-24 09:47:17,2017-07-24 09:47:18,NaN,201359382.170724093002.Table.1.selections.txt,1.0,VA,NaN,NaN,NaN,BSM_20170724_09471700.pt,0,BSM_2017-07-24
1,BSM_20170724_09471800.wav,0,0,0,0,0,1.0,evaluation_v2,NaN,1.0,2.0,a,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017\Boat,201359382,-172.7,2017-07-24 09:47:18,2017-07-24 09:47:19,NaN,201359382.170724093002.Table.1.selections.txt,1.0,VA,NaN,NaN,NaN,BSM_20170724_09471800.pt,1,BSM_2017-07-24


In [23]:
labels_df[labels_df["labeling_effort"] == "evaluation_v1"]["labeled_snippet_filename"].value_counts(dropna=False)

labeled_snippet_filename
201359382.170724133858.snippet.wav    600
201359382.170724140912.snippet.wav    600
201359382.210714080018.snippet.wav    600
201359382.210714083053.snippet.wav    600
5725.200726190001.snippet.wav         600
5725.200726191905.snippet.wav         354
5725.200726195504.snippet.wav         247
Name: count, dtype: int64

In [28]:
labels_df.groupby("labeled_snippet_filename")["Boat"].value_counts(dropna=False)

labeled_snippet_filename            Boat
201359382.170724094717.snippet.wav  1.0     120
201359382.170724103928.snippet.wav  1.0     120
201359382.170724133858.snippet.wav  1.0     600
201359382.170724140912.snippet.wav  0.0     600
201359382.170725060901.snippet.wav  1.0     120
201359382.210714080018.snippet.wav  0.0     600
201359382.210714083053.snippet.wav  1.0     600
201359382.210714135951.snippet.wav  0.0     116
201359382.210714142330.snippet.wav  0.0     120
201359382.210718085424.snippet.wav  1.0     120
201359382.210718085704.snippet.wav  1.0     120
201359382.210718182950.snippet.wav  0.0     116
201359382.210718192740.snippet.wav  1.0     120
201359382.210725161105.snippet.wav  0.0     120
201359382.210803184644.snippet.wav  0.0     120
201359382.210803191210.snippet.wav  0.0     120
201359382.210804123124.snippet.wav  0.0     120
201359382.210804124936.snippet.wav  0.0     120
201359382.210804132905.snippet.wav  1.0     120
5725.200726190001.snippet.wav       0.0     600

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

def create_test_fold_indices(
    labels_df: pd.DataFrame,
    *,
    n_splits: int = 5,
    stratify_col: str = "Site",
    group_col: str | None = None,  # <-- optional
    random_state: int = 42,
    out_col: str = "test_fold_idx",
) -> pd.DataFrame:
    """
    If group_col is provided:
      - non-NaN groups are kept together (avoid leakage) using StratifiedGroupKFold if available
      - NaN rows are assigned individually (stratified-ish) while balancing fold sizes
    If group_col is None:
      - simple row-level StratifiedKFold (your original behavior)
    """
    df = labels_df.copy()
    df[out_col] = -1

    if stratify_col not in df.columns:
        raise KeyError(f"stratify_col='{stratify_col}' not found in df.columns")
    if group_col is not None and group_col not in df.columns:
        raise KeyError(f"group_col='{group_col}' not found in df.columns")

    rng = np.random.RandomState(random_state)

    # ----------------------------
    # Case A: no grouping -> original behavior
    # ----------------------------
    if group_col is None:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        for fold_idx, (_, test_idx) in enumerate(skf.split(df, df[stratify_col])):
            df.iloc[test_idx, df.columns.get_loc(out_col)] = fold_idx
        return df

    # ----------------------------
    # Case B: grouping enabled
    # ----------------------------
    mask_grouped = df[group_col].notna()
    df_g = df[mask_grouped]
    df_n = df[~mask_grouped]

    # 1) Grouped (non-NaN)
    if len(df_g) > 0:
        try:
            from sklearn.model_selection import StratifiedGroupKFold

            sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            for fold_idx, (_, test_idx_local) in enumerate(
                sgkf.split(df_g, y=df_g[stratify_col].values, groups=df_g[group_col].values)
            ):
                df.loc[df_g.iloc[test_idx_local].index, out_col] = fold_idx

        except Exception:
            # Greedy fallback (balances class + total size)
            y = df_g[stratify_col].astype(str).values
            groups = df_g[group_col].astype(str).values
            classes = np.unique(y)

            group_to_idx = {}
            group_to_counts = {}
            for idx, (g, c) in zip(df_g.index, zip(groups, y)):
                group_to_idx.setdefault(g, []).append(idx)
                if g not in group_to_counts:
                    group_to_counts[g] = {cl: 0 for cl in classes}
                group_to_counts[g][c] += 1

            total_counts = {cl: int((y == cl).sum()) for cl in classes}
            target_per_fold = {cl: total_counts[cl] / n_splits for cl in classes}
            target_size = len(df_g) / n_splits

            fold_counts = [{cl: 0 for cl in classes} for _ in range(n_splits)]
            fold_sizes = [0 for _ in range(n_splits)]

            group_list = list(group_to_idx.keys())
            group_list.sort(key=lambda g: len(group_to_idx[g]), reverse=True)

            # shuffle ties deterministically
            i = 0
            while i < len(group_list):
                j = i
                sz = len(group_to_idx[group_list[i]])
                while j < len(group_list) and len(group_to_idx[group_list[j]]) == sz:
                    j += 1
                if j - i > 1:
                    chunk = group_list[i:j]
                    rng.shuffle(chunk)
                    group_list[i:j] = chunk
                i = j

            def fold_cost(k, g_counts, g_size):
                cost = 0.0
                for cl in classes:
                    new_val = fold_counts[k][cl] + g_counts[cl]
                    cost += (new_val - target_per_fold[cl]) ** 2
                new_size = fold_sizes[k] + g_size
                cost += 0.25 * (new_size - target_size) ** 2
                return cost

            for g in group_list:
                g_idx = group_to_idx[g]
                g_size = len(g_idx)
                g_counts = group_to_counts[g]
                best_fold = min(range(n_splits), key=lambda k: fold_cost(k, g_counts, g_size))

                df.loc[g_idx, out_col] = best_fold
                fold_sizes[best_fold] += g_size
                for cl in classes:
                    fold_counts[best_fold][cl] += g_counts[cl]

    # 2) NaN rows: assign individually, balancing final fold sizes
    if len(df_n) > 0:
        current_sizes = df[out_col].value_counts().to_dict()
        fold_sizes = [int(current_sizes.get(k, 0)) for k in range(n_splits)]

        y_n = df_n[stratify_col].astype(str)
        for cl in y_n.unique():
            idxs = df_n.index[y_n == cl].to_numpy()
            idxs = np.array(idxs, copy=True)  # <-- FIX: ensure writable for shuffle
            rng.shuffle(idxs)

            for idx in idxs:
                min_size = min(fold_sizes)
                candidates = [k for k, sz in enumerate(fold_sizes) if sz == min_size]
                chosen = candidates[rng.randint(len(candidates))]
                df.loc[idx, out_col] = chosen
                fold_sizes[chosen] += 1

    if (df[out_col] == -1).any():
        missing = int((df[out_col] == -1).sum())
        raise RuntimeError(f"{missing} rows were not assigned a fold (still -1).")

    return df

labels_df = create_test_fold_indices(labels_df, n_splits=5)

In [31]:
labels_df.groupby("test_fold_idx")["Boat"].value_counts(dropna=False)

test_fold_idx  Boat
0              NaN     1340
               1.0      898
               0.0      866
1              NaN     1361
               1.0      880
               0.0      863
2              NaN     1382
               0.0      876
               1.0      846
3              NaN     1380
               1.0      867
               0.0      856
4              NaN     1342
               0.0      897
               1.0      864
Name: count, dtype: int64